# 07. PatchTST: Transformers for Time Series via Patching

## Background

Applying standard transformers to time series hit a fundamental problem: attention complexity is O(L²) in sequence length, and time series with thousands of time steps are computationally prohibitive. More fundamentally, point-wise tokenization (treating each time step as one token) loses local temporal structure that is informationally rich.

PatchTST (Nie et al., 2023) solved this elegantly by borrowing ViT's (Vision Transformer) patch tokenization: segment the time series into fixed-length, optionally overlapping patches, and treat each patch as a token. This:
1. Reduces sequence length by patch_size, making attention tractable
2. Preserves local semantic structure within each patch
3. Enables longer context windows (the 'history' transformer models can see)

PatchTST also introduced **channel independence**: each univariate series is processed independently, sharing a single model — dramatically improving long-term forecasting accuracy on multivariate benchmarks despite ignoring cross-variable correlations.

## What You'll Learn

- Patch tokenization: stride, patch length, padding
- Channel independence vs channel mixing architectures
- PatchTST encoder architecture
- Why patching improves long-horizon forecasting

## Why This Matters

PatchTST established patching as the dominant paradigm for time series transformers, influencing iTransformer, TimesNet, and Chronos. Understanding why point-wise tokenization fails and how patches fix it is key to architecting any modern time series transformer for production.


## Patch Tokenization

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

def create_patches(x: torch.Tensor, patch_len: int, stride: int) -> torch.Tensor:
    '''Segment time series into overlapping patches.
    x: (B, T, C) — batch, time, channels
    Returns: (B, N_patches, patch_len, C)
    where N_patches = (T - patch_len) // stride + 1
    '''
    B, T, C = x.shape
    n_patches = (T - patch_len) // stride + 1

    patches = []
    for i in range(n_patches):
        start = i * stride
        end = start + patch_len
        patches.append(x[:, start:end, :])  # (B, patch_len, C)

    return torch.stack(patches, dim=1)  # (B, N, patch_len, C)

# Demonstrate patch tokenization
B, T, C = 4, 336, 1  # batch, look-back window, channels
x = torch.randn(B, T, C)

PATCH_LEN = 16
STRIDE = 8
patches = create_patches(x, PATCH_LEN, STRIDE)
N = patches.shape[1]

print(f'Patch Tokenization:')
print(f'  Input:         {tuple(x.shape)} (B, T, C)')
print(f'  Patch length:  {PATCH_LEN}')
print(f'  Stride:        {STRIDE}')
print(f'  N patches:     {N}')
print(f'  Patches shape: {tuple(patches.shape)}')
print(f'\nAttention complexity comparison:')
print(f'  Point-wise (L={T}): O({T}^2) = {T**2:,} attention operations')
print(f'  Patched (N={N}):   O({N}^2) = {N**2:,} attention operations')
print(f'  Reduction: {T**2//N**2}x fewer operations')


## PatchTST Encoder

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_len: int, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Linear(patch_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, patches: torch.Tensor) -> torch.Tensor:
        # patches: (B, N, patch_len, C) — flatten last dim, treat C independently
        B, N, P, C = patches.shape
        patches = patches.permute(0, 3, 1, 2)  # (B, C, N, P)
        patches = patches.reshape(B*C, N, P)    # (B*C, N, P)
        return self.dropout(self.proj(patches)) # (B*C, N, d_model)

class PatchTST(nn.Module):
    def __init__(self, context_len: int, horizon: int,
                 patch_len: int = 16, stride: int = 8,
                 d_model: int = 128, n_heads: int = 8,
                 n_layers: int = 3, dropout: float = 0.1,
                 n_channels: int = 1):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
        self.n_channels = n_channels
        self.n_patches = (context_len - patch_len) // stride + 1

        # Patch embedding (channel independent)
        self.patch_embed = PatchEmbedding(patch_len, d_model, dropout)

        # Positional encoding (learnable)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Forecast head: flatten patch embeddings -> project to horizon
        self.head = nn.Linear(self.n_patches * d_model, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, C)
        B, T, C = x.shape

        # Patching
        patches = create_patches(x, self.patch_len, self.stride)  # (B, N, P, C)

        # Embed: channel independence -> treat as B*C batch
        emb = self.patch_embed(patches)   # (B*C, N, d_model)
        emb = emb + self.pos_embed        # positional info

        # Transformer
        encoded = self.encoder(emb)       # (B*C, N, d_model)

        # Head
        flat = encoded.reshape(B*C, -1)   # (B*C, N*d_model)
        forecast = self.head(flat)        # (B*C, horizon)

        return forecast.reshape(B, C, -1).permute(0, 2, 1)  # (B, horizon, C)

CONTEXT, HORIZON, CHANNELS = 336, 96, 1
model = PatchTST(CONTEXT, HORIZON, patch_len=16, stride=8,
                  d_model=128, n_heads=8, n_layers=3, n_channels=CHANNELS)

params = sum(p.numel() for p in model.parameters())
print(f'PatchTST: {params:,} parameters')
print(f'N patches: {model.n_patches}')

x = torch.randn(8, CONTEXT, CHANNELS)
y = model(x)
print(f'Input: {tuple(x.shape)}, Output: {tuple(y.shape)}')
